# Unit 2 Case Study: Refactoring a Shopping Cart to the SOLID Principles

Ian Stevens, Advanced Object-Oriented Design and Programming, Unit 2.

The case study asks for a simple online shopping system in which users add products to a cart, the total is calculated and discounts are applied. It starts from a poorly designed `Order` class and refactors it to follow the five SOLID principles. This notebook runs both versions side by side:

* `shop_initial.py` keeps the brief's code, with an `Item` class and a discount method added so that it runs and meets the requirement.
* `shop_solid.py` is the refactored design.
* `test_shop.py` holds 38 tests and the doctests from both modules: `python -m unittest -v test_shop`.

Needs Python 3.10 or later and no third-party packages. Section 5 uses mypy, which is optional.

In [1]:
import contextlib
import inspect
import io
import platform
from decimal import Decimal

import shop_initial
import shop_solid
from shop_solid import (CreditCardPayment, CryptoPayment, Discount, FixedAmountDiscount,
                        Order, PaymentMethod, PaymentProcessor, PayPalPayment,
                        PercentageDiscount, Product, Refundable, RefundService)

print("Python", platform.python_version(), platform.python_implementation())

Python 3.13.18 CPython


## 1. The initial design

The brief lists three problems: `Order` does too much (SRP), a new payment method means editing `Order` (OCP), and `Order` is tied to the concrete payment methods (DIP). Running the code turned up three more.

In [2]:
order = shop_initial.Order()
order.add_item(shop_initial.Item("Notebook", 0.10))
order.add_item(shop_initial.Item("Pencil", 0.20))
print("Total:", order.calculate_total())

with contextlib.redirect_stdout(io.StringIO()) as captured:
    returned = order.pay("credit")
print("pay('credit') printed", repr(captured.getvalue()), "and returned", returned)

with contextlib.redirect_stdout(io.StringIO()) as captured:
    returned = order.pay("applepay")
print("pay('applepay') printed", repr(captured.getvalue()), "and returned", returned)

cheap = shop_initial.Order()
cheap.add_item(shop_initial.Item("Pencil", 2.00))
cheap.apply_discount("fixed", 5)
print("Pencil with £5 off:", cheap.calculate_total())

Total: 0.30000000000000004
pay('credit') printed 'Processing credit card payment...\n' and returned None
pay('applepay') printed '' and returned None
Pencil with £5 off: -3.0


Floats give a total of 0.30000000000000004. `pay()` prints and returns `None`, so a caller cannot tell whether a payment went through, and the only way to test it is to capture standard output. An unknown payment type does nothing at all, which in a real shop would mean an order marked as paid with no money taken. A fixed discount larger than the order gives a negative total.

Each `if` and `elif` in `Order` is a decision that will need another branch when the business changes. Counting them, and the responsibilities they belong to, shows how many reasons `Order` has to change.

In [3]:
import ast

def branches(cls):
    tree = ast.parse(inspect.getsource(cls))
    return sum(isinstance(node, ast.If) for node in ast.walk(tree))

methods = [name for name, _ in inspect.getmembers(shop_initial.Order, inspect.isfunction)
           if not name.startswith("_")]
print("Order's public methods:", methods)
print("if/elif branches in Order:", branches(shop_initial.Order))

Order's public methods: ['add_item', 'apply_discount', 'calculate_total', 'pay']
if/elif branches in Order: 4


## 2. Single Responsibility: splitting `Order`

The brief states SRP as a class having only one reason to change. In the original, cart rules, pricing rules, discount rules and payment providers all change for different reasons and at different times. The refactor gives each its own class. `Order` keeps items and totals, `Discount` subclasses work out reductions, `PaymentMethod` subclasses take payments, and `PaymentProcessor` puts them together at checkout.

Money is now a `Decimal` rounded to the penny, so the total is exact.

In [4]:
order = Order()
order.add_item(Product("NB-01", "Notebook", "4.50"), 2)
order.add_item(Product("PN-02", "Pen", "1.20"))
for product, quantity in order.items:
    print(f"{quantity} x {product.name:<8} £{product.price}")
print("Subtotal: £", order.subtotal(), sep="")
print("Anything about payment on Order?", [n for n in dir(Order) if "pay" in n.lower()])

2 x Notebook £4.50
1 x Pen      £1.20
Subtotal: £10.20
Anything about payment on Order? []


## 3. Open/Closed: new payment methods and discounts as new classes

`PaymentProcessor` is written against the `PaymentMethod` abstraction and never names a concrete class. `CryptoPayment` was written after the card and PayPal classes, and adding it changed nothing that already existed. The same is true of discounts. A new kind of discount, here written inside the notebook, works without touching `Order`.

In [5]:
class StudentDiscount(Discount):
    """15% off, capped at £5. Defined here, outside shop_solid.py."""
    def amount_off(self, subtotal):
        return min(subtotal * Decimal("0.15"), Decimal("5.00"))

for discount in (None, PercentageDiscount(10), FixedAmountDiscount("2.50"),
                 FixedAmountDiscount(50), StudentDiscount()):
    name = type(discount).__name__ if discount else "none"
    print(f"{name:<20} total £{order.total(discount)}")

none                 total £10.20
PercentageDiscount   total £9.18
FixedAmountDiscount  total £7.70
FixedAmountDiscount  total £0.00
StudentDiscount      total £8.67


In [6]:
source = inspect.getsource(PaymentProcessor)
print("Concrete payment classes named in PaymentProcessor:",
      [c for c in ("CreditCard", "PayPal", "Crypto") if c in source] or "none")
print("if/elif branches: initial Order", branches(shop_initial.Order),
      "| refactored Order", branches(Order),
      "| PaymentProcessor", branches(PaymentProcessor))

Concrete payment classes named in PaymentProcessor: none
if/elif branches: initial Order 4 | refactored Order 5 | PaymentProcessor 3


Counting branches turned out to be a poor measure. The refactored `Order` has five, one more than the original, because it now checks quantities, refuses a SKU that clashes with a different product and checks every discount's result. What has gone is the kind of branch that grows: none of the branches in the new design asks which payment method or discount is in use, so adding one does not add a branch.

## 4. Liskov Substitution: one contract for every payment method

`PaymentMethod.pay()` states a contract: it accepts any amount above zero, returns a `PaymentResult` for that amount, and reports a decline in the result rather than raising. `pay()` is a template method. The base class checks the precondition and then calls the subclass's `_take_payment()`, so a subclass cannot quietly change what counts as a valid amount. The same checks run against every subclass, and `PaymentProcessor` gives the same answer whichever one it is handed.

In [7]:
methods = [CreditCardPayment("4111 1111 1111 1111"),
           PayPalPayment("sam@example.com"),
           CryptoPayment("0xA1B2C3D4E5", "50000")]

for method in methods:
    result = PaymentProcessor(method).checkout(order, PercentageDiscount(10))
    print(f"{method.description:<28} succeeded={result.succeeded} £{result.amount} "
          f"ref={result.reference} ({result.message})")

credit card ending 1111      succeeded=True £9.18 ref=CRED-000001 (card payment authorised)
PayPal (s***@example.com)    succeeded=True £9.18 ref=PAYP-000002 (PayPal payment completed)
crypto wallet 0xA1B2...      succeeded=True £9.18 ref=CRYP-000003 (sent 0.00018360 coins)


The template method only protects the precondition if subclasses leave `pay()` alone, and Python cannot stop them overriding it. The next class does exactly that, refusing orders under £10. It is a perfectly reasonable business rule, but expressed this way it strengthens the precondition, and code that worked with every other payment method now fails.

In [8]:
class MinimumSpendCardPayment(CreditCardPayment):
    def pay(self, amount):
        if amount < Decimal("10.00"):
            raise ValueError("minimum card spend is £10")
        return super().pay(amount)

small = Order()
small.add_item(Product("PN-02", "Pen", "1.20"))
for method in (CreditCardPayment("4111111111111111"), MinimumSpendCardPayment("4111111111111111")):
    try:
        result = PaymentProcessor(method).checkout(small)
        print(type(method).__name__, "->", result.succeeded, result.message)
    except ValueError as error:
        print(type(method).__name__, "-> ValueError:", error)

CreditCardPayment -> True card payment authorised
MinimumSpendCardPayment -> ValueError: minimum card spend is £10


The substitutable version keeps the rule but reports it the way the contract allows, as a declined `PaymentResult` from `_take_payment()`. Callers already handle a decline, so nothing else has to change.

In [9]:
class MinimumSpendCard(CreditCardPayment):
    def _take_payment(self, amount):
        if amount < Decimal("10.00"):
            return self._result(False, amount, "declined: minimum card spend is £10")
        return super()._take_payment(amount)

result = PaymentProcessor(MinimumSpendCard("4111111111111111")).checkout(small)
print(result.succeeded, result.message)

False declined: minimum card spend is £10


## 5. Interface Segregation: refunds are a separate role

Card and PayPal payments can be refunded. A crypto transfer cannot. If `refund()` were on `PaymentMethod`, `CryptoPayment` would have to implement it by raising an error, which is the kind of forced, empty implementation ISP warns about and would also break LSP. Instead, `Refundable` is a small protocol of its own, and `RefundService` depends only on it.

In [10]:
card = CreditCardPayment("4111 1111 1111 1111")
paid = PaymentProcessor(card).checkout(order)
print("Paid:", paid.reference, "£", paid.amount)

refunds = RefundService(card)
for amount in ("4.50", "6.00"):
    r = refunds.refund(paid.reference, amount)
    print(f"Refund £{amount}: succeeded={r.succeeded} {r.message}")

for method in methods:
    print(f"{type(method).__name__:<18} Refundable? {isinstance(method, Refundable)}")

try:
    RefundService(CryptoPayment("0xA1B2C3D4E5", "50000"))
except TypeError as error:
    print("TypeError:", error)

Paid: CRED-000006 £ 10.20
Refund £4.50: succeeded=True refund against CRED-000006
Refund £6.00: succeeded=False refund exceeds the amount still held
CreditCardPayment  Refundable? True
PayPalPayment      Refundable? True
CryptoPayment      Refundable? False
TypeError: CryptoPayment does not support refunds


The runtime check is a backstop. A static type checker catches the same mistake before the code runs. With mypy installed, checking a file that passes a `CryptoPayment` to `RefundService` gives:

```
error: Argument 1 to "RefundService" has incompatible type "CryptoPayment"; expected "Refundable"  [arg-type]
```

`shop_solid.py` itself passes `mypy --strict` with no errors.

## 6. Dependency Inversion: the policy depends on abstractions

`PaymentProcessor` is the high-level policy: total the order, apply the discount, take payment. It receives its `PaymentMethod` through its constructor and never creates one, so the decision about which concrete class to use sits with the caller. This is also what makes it easy to test. A test double that records what it was asked to charge stands in for a real provider.

In [11]:
class RecordingPayment(PaymentMethod):
    description = "test double"
    def __init__(self):
        self.charged = []
    def _take_payment(self, amount):
        self.charged.append(amount)
        return self._result(True, amount, "recorded")

double = RecordingPayment()
PaymentProcessor(double).checkout(order, FixedAmountDiscount("0.20"))
print("Charged:", double.charged)

Charged: [Decimal('10.00')]


In [12]:
processor_refs = {n.id for n in ast.walk(ast.parse(inspect.getsource(PaymentProcessor)))
                  if isinstance(n, ast.Name) and n.id in shop_solid.__all__}
order_refs = {n.id for n in ast.walk(ast.parse(inspect.getsource(Order)))
              if isinstance(n, ast.Name) and n.id in shop_solid.__all__}
print("PaymentProcessor refers to:", sorted(processor_refs))
print("Order refers to:           ", sorted(order_refs))

PaymentProcessor refers to: ['Discount', 'Order', 'PaymentMethod', 'PaymentResult']
Order refers to:            ['Discount', 'NoDiscount', 'Product']


`PaymentProcessor` refers to the `PaymentMethod` and `Discount` abstractions, to `Order`, and to the `PaymentResult` value it returns, but to none of the concrete payment or discount classes. `Order` and `PaymentResult` are concrete, which a strict reading of DIP would object to. I left them that way on purpose: both are stable data holders rather than volatile providers, which is the distinction Marcotte (2022, ch. 7) draws between stable and volatile dependencies. `Order` refers only to `Product` and the `Discount` family.

One point in the brief needed thought. Its DIP step says that the high-level `Order` should stop depending on the low-level `PaymentProcessor`. After the SRP step, though, `Order` has no dependency on payment at all. The dependency that needs inverting runs the other way, from the processor, which holds the checkout policy, to the payment providers. That is the one this design turns round.

## 7. Running the tests

In [13]:
import unittest
import test_shop

suite = unittest.defaultTestLoader.loadTestsFromModule(test_shop)
result = unittest.TextTestRunner(stream=io.StringIO(), verbosity=0).run(suite)
print(f"Ran {result.testsRun} tests: {len(result.failures)} failures, {len(result.errors)} errors")

Ran 38 tests: 0 failures, 0 errors
